# Code Ownership, notebook 4: hyperparameter tuning without selection bias

Every encoder x feature set x head is tuned inside each outer training set by 4 inner folds grouped by function; the outer fold's prediction comes from a head refit on the
whole outer training set. Grids are in `helpers/ko_models.py` (fixed before the run). The runner is resumable and takes a while; finished combinations are kept in
`results/tuning_preds.pkl`.

In [ ]:
import sys, warnings, json, pickle
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../../helpers").resolve()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
import ko_data as K, ko_models as M
from core import metrics, group_bootstrap, paired_diff_bootstrap, qwk, band
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = K.load()
dev = d[d.role == "dev"].reset_index(drop=True)
test = d[d.role == "test"].reset_index(drop=True)
y, groups, folds = dev.score.to_numpy(), dev.snippet_id.to_numpy(), dev.fold.to_numpy()
def ci(p, stat):
    return group_bootstrap(y, p, groups, stat, n=1000)
def sp(a, b):
    return spearmanr(a, b).correlation
def mae(a, b):
    return float(np.mean(np.abs(a - b)))
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [ ]:
import run_tuning
from embedder import Encoder
ENCODERS = [k for k in ("unixcoder", "codebert", "mpnet") if (Encoder._local(k) or k == "mpnet")]
P = run_tuning.run(ENCODERS, workers=10)
print(len(P), "combinations")

In [ ]:
rows = []
for key, v in P.items():
    e, s, h = key.split("|"); p = v["pred"]
    rows.append({"encoder": e, "feature set": s, "head": h, "RMSE": rmse(p, y), "MAE": mae(p, y), "Spearman": sp(p, y)})
T = pd.DataFrame(rows)
T.to_csv(RESULTS / "tuning.csv", index=False)
T.pivot_table(index=["encoder", "feature set"], columns="head", values="RMSE").round(3)

In [ ]:
fig, axes = plt.subplots(1, len(ENCODERS), figsize=(5 * len(ENCODERS), 4), sharey=True, squeeze=False)
for ax, e in zip(axes[0], ENCODERS):
    sns.heatmap(T[T.encoder == e].pivot(index="feature set", columns="head", values="RMSE"), annot=True, fmt=".2f", cmap="viridis_r", ax=ax, cbar=False); ax.set_title(e)
plt.tight_layout(); plt.show()

Settings chosen in each outer fold for the ridge heads (stability of the choice):

In [ ]:
ch = []
for key, v in P.items():
    e, s, h = key.split("|")
    if h == "ridge":
        ch.append({"encoder": e, "feature set": s, **{f"fold {f}": p["alpha"] for f, p in v["chosen"].items()}})
pd.DataFrame(ch)

## Reading the results

_Reading of the results: written after the run._